In [ ]:
## Notebook 3 — Jointures et agrégations
##💡 Créer un notebook 03_transformations.ipynb. Utiliser les DataFrames nettoyés du jalon 2.



In [5]:
## Initialisation session Spark et chargement des données
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Notebook 3 - Transformations") \
    .getOrCreate()

PATH = "/home/jovyan/data/tmp"

df_customers = spark.read.parquet(f"{PATH}/customers")
df_employees = spark.read.parquet(f"{PATH}/employees")
df_orders = spark.read.parquet(f"{PATH}/orders")
df_order_details = spark.read.parquet(f"{PATH}/order_details")
df_products = spark.read.parquet(f"{PATH}/products")

print("Toutes les tables ont été chargées avec succès !")
## Verification
tables = {
    "customers": df_customers,
    "employees": df_employees,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products
}

for name, df in tables.items():
    print(f"Table '{name}' : {df.count()} lignes chargées.")


Toutes les tables ont été chargées avec succès !
Table 'customers' : 91 lignes chargées.
Table 'employees' : 9 lignes chargées.
Table 'orders' : 408 lignes chargées.
Table 'order_details' : 2155 lignes chargées.
Table 'products' : 66 lignes chargées.


In [6]:
## Q21 — Jointure orders + customers - Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country, order_date, freight.

from pyspark.sql.functions import col

df_orders_customers = df_orders.join(
    df_customers, 
    on="customer_id", 
    how="inner"
).select(
    col("order_id"),
    col("company_name"),
    col("country"),
    col("order_date"),
    col("freight")
)

df_orders_customers.show(5, truncate=False)


+--------+----------------------------+-------+----------+-------+
|order_id|company_name                |country|order_date|freight|
+--------+----------------------------+-------+----------+-------+
|10400   |Eastern Connection          |UK     |1997-01-01|83.93  |
|10401   |Rattlesnake Canyon Grocery  |USA    |1997-01-01|12.51  |
|10402   |Ernst Handel                |AUSTRIA|1997-01-02|67.88  |
|10403   |Ernst Handel                |AUSTRIA|1997-01-03|73.79  |
|10404   |Magazzini Alimentari Riuniti|ITALY  |1997-01-03|155.97 |
+--------+----------------------------+-------+----------+-------+
only showing top 5 rows



In [8]:
## Q22 — Jointure order_details + products - Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id, unit_price depuis products.

from pyspark.sql.functions import col

df_order_details_products = df_order_details.join(
    df_products,
    on="product_id",
    how="inner"
).select(
    col("order_id"),
    col("product_id"),
    col("unit_price"),
    col("quantite"), 
    col("discount"),
    col("product_name"),
    col("category_id")
)

df_order_details_products.show(5, truncate=False)

+--------+----------+----------+--------+--------+-------------------------------+-----------+
|order_id|product_id|unit_price|quantite|discount|product_name                   |category_id|
+--------+----------+----------+--------+--------+-------------------------------+-----------+
|10248   |11        |21.0      |12      |0.0     |Queso Cabrales                 |4          |
|10248   |72        |34.8      |5       |0.0     |Mozzarella di Giovanni         |4          |
|10249   |14        |23.25     |9       |0.0     |Tofu                           |7          |
|10249   |51        |53.0      |40      |0.0     |Manjimup Dried Apples          |7          |
|10250   |41        |9.65      |10      |0.0     |Jack's New England Clam Chowder|8          |
+--------+----------+----------+--------+--------+-------------------------------+-----------+
only showing top 5 rows



In [12]:
## Q23 — Jointure products + categories - Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et description.

from pyspark.sql.functions import col


df_categories = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/raw/categories.csv")

df_categories.write.mode("overwrite").parquet(f"{PATH}/categories")

df_categories = spark.read.parquet(f"{PATH}/categories")

# 4. Q23 — Jointure df_products et df_categories sur category_id
df_products_categories = df_products.join(
    df_categories,
    on="category_id",
    how="inner"
).select(
    col("product_id"),
    col("product_name"),
    col("category_id"),
    col("category_name"),
    col("description"),
    col("unit_price")
)

df_products_categories.show(5, truncate=False)
!ls -l /home/jovyan/data/tmp

+----------+-------------------------------+-----------+-------------+----------------------------------------------------------+----------+
|product_id|product_name                   |category_id|category_name|description                                               |unit_price|
+----------+-------------------------------+-----------+-------------+----------------------------------------------------------+----------+
|3         |Aniseed Syrup                  |2          |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|10.0      |
|4         |Chef Anton's Cajun Seasoning   |2          |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|22.0      |
|6         |Grandma's Boysenberry Spread   |2          |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|25.0      |
|7         |Uncle Bob's Organic Dried Pears|7          |Produce      |Dried fruit and bean curd                                 |30.0      |
|8         |N

In [16]:
## Q24 — DataFrame enrichi complet
## A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec categories, employees, shippers) sans renommer aucune colonne. 
## Lister ensuite les colonnes qui apparaissent en double grâce à Counter.

from collections import Counter
from pyspark.sql.functions import col

# 1. Chargement et préparation des tables avec renommage préfixé pour éviter les doublons
df_cust_renamed = df_customers \
    .withColumnRenamed("country", "customer_country") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("phone", "customer_phone")

df_emp_renamed = df_employees \
    .withColumnRenamed("country", "employee_country") \
    .withColumnRenamed("city", "employee_city") \
    .withColumnRenamed("unit_price", "employee_unit_price") # si besoin

try:
    df_ship = spark.read.parquet(f"{PATH}/shippers")
except:
    df_ship = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/raw/shippers.csv")
    df_ship.write.mode("overwrite").parquet(f"{PATH}/shippers")
    df_ship = spark.read.parquet(f"{PATH}/shippers")

df_ship_renamed = df_ship.withColumnRenamed("company_name", "shipper_name")

df_prod_renamed = df_products.withColumnRenamed("unit_price", "product_unit_price")
df_cat_renamed = df_categories.withColumnRenamed("description", "category_description")

df_prod_full = df_prod_renamed.join(df_cat_renamed, on="category_id", how="inner")

# 2. Re-jointure complète avec les tables renommées
df_orders_enriched = df_order_details \
    .join(df_orders, on="order_id", how="inner") \
    .join(df_cust_renamed, on="customer_id", how="inner") \
    .join(df_emp_renamed, on="employee_id", how="inner") \
    .join(df_ship_renamed, on="shipper_id", how="inner") \
    .join(df_prod_full, on="product_id", how="inner")

# 3. Vérification des doublons restants
col_counts = Counter(df_orders_enriched.columns)
remaining_duplicates = [c for c, count in col_counts.items() if count > 1]

print("Doublons restants :", remaining_duplicates)
print("Nombre total de colonnes :", len(df_orders_enriched.columns))

## Verif


# 1. Chargement propre de shippers si nécessaire
try:
    df_shippers = spark.read.parquet(f"{PATH}/shippers")
except:
    df_shippers = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/raw/shippers.csv")
    df_shippers.write.mode("overwrite").parquet(f"{PATH}/shippers")
    df_shippers = spark.read.parquet(f"{PATH}/shippers")

# 2. Reconstitution de la jointure brute (Q24a) avec on="shipper_id"
df_products_enriched_raw = df_products.join(df_categories, on="category_id", how="inner")

df_full_raw = df_order_details \
    .join(df_orders, on="order_id", how="inner") \
    .join(df_customers, on="customer_id", how="inner") \
    .join(df_employees, on="employee_id", how="inner") \
    .join(df_shippers, on="shipper_id", how="inner") \
    .join(df_products_enriched_raw, on="product_id", how="inner")

# 3. Analyse des occurrences de chaque nom de colonne avec Counter
col_counts = Counter(df_full_raw.columns)

print("--- Détail des colonnes en double (Q24a) ---")
for col_name, count in col_counts.items():
    if count > 1:
        print(f"Colonne '{col_name}' : présente {count} fois")


Doublons restants : []
Nombre total de colonnes : 52
--- Détail des colonnes en double (Q24a) ---
Colonne 'company_name' : présente 2 fois
Colonne 'city' : présente 2 fois
Colonne 'country' : présente 2 fois
Colonne 'phone' : présente 2 fois


In [18]:
## ## Q24 — DataFrame enrichi complet
## B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
## en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
## df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon.

# Q24b — Renommage préfixé des tables d'origine pour éliminer les doublons
df_cust_renamed = df_customers \
    .withColumnRenamed("country", "customer_country") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("phone", "customer_phone") \
    .withColumnRenamed("company_name", "customer_company_name")

df_emp_renamed = df_employees \
    .withColumnRenamed("country", "employee_country") \
    .withColumnRenamed("city", "employee_city")

df_ship_renamed = df_shippers \
    .withColumnRenamed("company_name", "shipper_name") \
    .withColumnRenamed("phone", "shipper_phone")

df_prod_renamed = df_products.withColumnRenamed("unit_price", "product_unit_price")
df_cat_renamed = df_categories.withColumnRenamed("description", "category_description")

df_prod_full = df_prod_renamed.join(df_cat_renamed, on="category_id", how="inner")

# Reconstruction de df_orders_enriched avec les tables renommées
df_orders_enriched = df_order_details \
    .join(df_orders, on="order_id", how="inner") \
    .join(df_cust_renamed, on="customer_id", how="inner") \
    .join(df_emp_renamed, on="employee_id", how="inner") \
    .join(df_ship_renamed, on="shipper_id", how="inner") \
    .join(df_prod_full, on="product_id", how="inner")

# Double vérification finale des doublons
remaining_counts = Counter(df_orders_enriched.columns)
print("Doublons restants après renommage :", [col for col, count in remaining_counts.items() if count > 1])
print("Nombre total de colonnes :", len(df_orders_enriched.columns))


Doublons restants après renommage : []
Nombre total de colonnes : 52


In [21]:
## Q25 — CA par client - Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA décroissant. Afficher le top 10
from pyspark.sql.functions import col, sum, desc

# Q25 — CA par client en utilisant directement le sous_total ou en castant les types
df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .groupBy("customer_company_name") \
    .agg(sum("ca").alias("ca_total")) \
    .orderBy(desc("ca_total")) \
    .show(10, truncate=False)


+----------------------------+-----------------+
|customer_company_name       |ca_total         |
+----------------------------+-----------------+
|QUICK-Stop                  |51682.73499999999|
|Save-a-lot Markets          |40238.08500000001|
|Ernst Handel                |39975.9075       |
|Mère Paillarde              |22871.06         |
|Rattlesnake Canyon Grocery  |17636.1          |
|Simons bistro               |16232.4125       |
|Hungry Owl All-Night Grocers|14403.025        |
|Folk och fä HB              |13200.92         |
|HILARION-Abastos            |11799.744        |
|Berglunds snabbköp          |11758.915        |
+----------------------------+-----------------+
only showing top 10 rows



In [22]:
## Q26 — CA par catégorie - Calculer le CA total par catégorie de produits. Afficher le nombre de produits distincts vendus par catégorie
from pyspark.sql.functions import col, sum, countDistinct, desc

# Q26 — CA total et produits distincts vendus par catégorie
df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .groupBy("category_name") \
    .agg(
        sum("ca").alias("ca_total"),
        countDistinct("product_id").alias("nb_produits_distincts")
    ) \
    .orderBy(desc("ca_total")) \
    .show(truncate=False)


+--------------+------------------+---------------------+
|category_name |ca_total          |nb_produits_distincts|
+--------------+------------------+---------------------+
|Dairy Products|108086.89000000001|9                    |
|Beverages     |90368.62999999999 |9                    |
|Confections   |82657.7505        |13                   |
|Seafood       |66959.2175        |12                   |
|Condiments    |54994.965         |11                   |
|Grains/Cereals|51463.625         |6                    |
|Produce       |40992.087499999994|4                    |
|Meat/Poultry  |11017.165         |2                    |
+--------------+------------------+---------------------+



In [23]:
## Q27 — CA par mois - Calculer le CA mensuel. Utiliser date_trunc ou month() et year() pour extraire le mois et l'année
from pyspark.sql.functions import col, sum, date_trunc, asc

# Q27 — CA mensuel par ordre chronologique
df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .withColumn("mois", date_trunc("month", col("order_date"))) \
    .groupBy("mois") \
    .agg(sum("ca").alias("ca_mensuel")) \
    .orderBy(asc("mois")) \
    .show(20, truncate=False)


+-------------------+------------------+
|mois               |ca_mensuel        |
+-------------------+------------------+
|1997-01-01 00:00:00|51487.5           |
|1997-02-01 00:00:00|31549.035         |
|1997-03-01 00:00:00|33226.32000000001 |
|1997-04-01 00:00:00|41510.60249999999 |
|1997-05-01 00:00:00|48895.265         |
|1997-06-01 00:00:00|29875.4525        |
|1997-07-01 00:00:00|45162.8575        |
|1997-08-01 00:00:00|38039.92999999999 |
|1997-09-01 00:00:00|43335.4025        |
|1997-10-01 00:00:00|48574.490000000005|
|1997-11-01 00:00:00|39898.784         |
|1997-12-01 00:00:00|54984.69149999999 |
+-------------------+------------------+



In [24]:
## Q28 — Performance par employé - Calculer pour chaque employé (full_name) : le nombre de commandes traitées, le CA total généré et le délai moyen de livraison en jours.
from pyspark.sql.functions import col, sum, count, avg, datediff, desc

# Q28 — Performance par employé (full_name, nb_commandes, ca_total, delai_moyen_jours)
df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .withColumn("delai_livraison", datediff(col("shipped_date"), col("order_date"))) \
    .groupBy("full_name") \
    .agg(
        count("order_id").alias("nb_commandes"),
        sum("ca").alias("ca_total"),
        avg("delai_livraison").alias("delai_moyen_jours")
    ) \
    .orderBy(desc("ca_total")) \
    .show(truncate=False)


+----------------+------------+------------------+------------------+
|full_name       |nb_commandes|ca_total          |delai_moyen_jours |
+----------------+------------+------------------+------------------+
|Margaret Peacock|182         |104193.74500000002|8.302197802197803 |
|Janet Leverling |165         |97081.25550000001 |8.921212121212122 |
|Nancy Davolio   |137         |81898.87749999999 |7.839416058394161 |
|Andrew Fuller   |84          |54907.02999999999 |10.19047619047619 |
|Robert King     |74          |49562.784999999996|9.81081081081081  |
|Laura Callahan  |103         |47077.94          |7.951456310679611 |
|Michael Suyama  |71          |34037.0625        |7.943661971830986 |
|Anne Dodsworth  |38          |20595.9925        |10.026315789473685|
|Steven Buchanan |39          |17185.642499999998|6.461538461538462 |
+----------------+------------+------------------+------------------+



In [25]:
## Q29 — Window functions — Rang - Classer les produits par CA généré avec dense_rank(). Utiliser une Window partitionnée par category_name.
## Indice : from pyspark.sql.window import Window / from pyspark.sql.functions import dense_rank /
## Window.partitionBy('category_name').orderBy(desc('ca'))

from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, dense_rank, desc

# Q29 — Classement des produits par CA au sein de chaque catégorie avec dense_rank()
df_produits_ca = df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .groupBy("category_name", "product_name") \
    .agg(sum("ca").alias("ca"))

window_spec = Window.partitionBy("category_name").orderBy(desc("ca"))

df_produits_ca.withColumn("rang", dense_rank().over(window_spec)) \
    .orderBy("category_name", "rang") \
    .show(30, truncate=False)


+-------------+--------------------------------+------------------+----+
|category_name|product_name                    |ca                |rang|
+-------------+--------------------------------+------------------+----+
|Beverages    |Côte de Blaye                   |49198.085         |1   |
|Beverages    |Ipoh Coffee                     |11069.9           |2   |
|Beverages    |Lakkalikööri                    |7379.1            |3   |
|Beverages    |Outback Lager                   |5468.4            |4   |
|Beverages    |Steeleye Stout                  |5274.9            |5   |
|Beverages    |Rhönbräu Klosterbier            |4485.545          |6   |
|Beverages    |Chartreuse verte                |4475.700000000001 |7   |
|Beverages    |Sasquatch Ale                   |2107.0            |8   |
|Beverages    |Laughing Lumberjack Lager       |910.0             |9   |
|Condiments   |Louisiana Fiery Hot Pepper Sauce|9373.185000000001 |1   |
|Condiments   |Sirop d'érable                  |909

In [26]:
## Q30 — Window functions — Cumul - Calculer le CA cumulé par mois (ordre chronologique) avec sum() sur une Window orderBy date.

from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, date_trunc, asc

# Q30 — CA mensuel et CA cumulé chronologique
df_ca_mois = df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .withColumn("mois", date_trunc("month", col("order_date"))) \
    .groupBy("mois") \
    .agg(sum("ca").alias("ca_mensuel"))

window_cumul = Window.orderBy(asc("mois"))

df_ca_mois.withColumn("ca_cumule", sum("ca_mensuel").over(window_cumul)) \
    .orderBy(asc("mois")) \
    .show(20, truncate=False)

+-------------------+------------------+------------------+
|mois               |ca_mensuel        |ca_cumule         |
+-------------------+------------------+------------------+
|1997-01-01 00:00:00|51487.5           |51487.5           |
|1997-02-01 00:00:00|31549.035         |83036.535         |
|1997-03-01 00:00:00|33226.32000000001 |116262.85500000001|
|1997-04-01 00:00:00|41510.60249999999 |157773.45750000002|
|1997-05-01 00:00:00|48895.265         |206668.72250000003|
|1997-06-01 00:00:00|29875.4525        |236544.17500000005|
|1997-07-01 00:00:00|45162.8575        |281707.03250000003|
|1997-08-01 00:00:00|38039.92999999999 |319746.9625       |
|1997-09-01 00:00:00|43335.4025        |363082.365        |
|1997-10-01 00:00:00|48574.490000000005|411656.855        |
|1997-11-01 00:00:00|39898.784         |451555.63899999997|
|1997-12-01 00:00:00|54984.69149999999 |506540.3305       |
+-------------------+------------------+------------------+



In [27]:
## Q31 — Tri et limite - Afficher les 5 produits les plus vendus en quantité (toutes commandes confondues).
## Afficher les 3 pays clients (customer_country) qui génèrent le plus de chiffre d'affaires.

from pyspark.sql.functions import col, sum, desc

# 1. Top 5 des produits les plus vendus en quantité
df_orders_enriched.groupBy("product_name") \
    .agg(sum("quantite").alias("total_quantite")) \
    .orderBy(desc("total_quantite")) \
    .show(5, truncate=False)

# 2. Top 3 des pays clients générant le plus de chiffre d'affaires
df_orders_enriched.withColumn("ca", col("quantite").cast("double") * col("prix_unitaire").cast("double") * (1 - col("discount").cast("double"))) \
    .groupBy("customer_country") \
    .agg(sum("ca").alias("ca_total")) \
    .orderBy(desc("ca_total")) \
    .show(3, truncate=False)

+----------------------+--------------+
|product_name          |total_quantite|
+----------------------+--------------+
|Gnocchi di nonna Alice|971           |
|Raclette Courdavault  |752           |
|Camembert Pierrot     |665           |
|Rhönbräu Klosterbier  |630           |
|Sir Rodney's Scones   |610           |
+----------------------+--------------+
only showing top 5 rows

+----------------+------------------+
|customer_country|ca_total          |
+----------------+------------------+
|GERMANY         |100641.26250000001|
|USA             |90731.68249999998 |
|AUSTRIA         |46559.4875        |
+----------------+------------------+
only showing top 3 rows



In [30]:
## Q32 — Écriture en Parquet - Écrire df_orders_enriched en format Parquet dans /home/jovyan/data/output/orders_enriched.parquet. Utiliser le mode overwrite.

df_orders_enriched.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/output/orders_enriched.parquet")

print("Écriture au format Parquet effectuée avec succès !")

# Vérificatio
# df_verify = spark.read.parquet("/home/jovyan/data/output/orders_enriched.parquet")

# print(f"Nombre de lignes lues : {df_verify.count()}")
# df_verify.show(5, truncate=False)

Écriture au format Parquet effectuée avec succès !
